<h2 style="color: Black;">Import The Necessary Library</h2>


In [13]:
import tensorflow as tf
import numpy as np### math computations
import matplotlib.pyplot as plt### plotting bar chart
import sklearn### machine learning library
import cv2## image processing
from sklearn.metrics import confusion_matrix, roc_curve### metrics
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer
from tensorflow.keras.layers import (GlobalAveragePooling2D, Activation, MaxPooling2D, Add, Conv2D, MaxPool2D, Dense,
                                     Flatten, InputLayer, BatchNormalization, Input, Embedding, Permute,
                                     Dropout, RandomFlip, RandomRotation, LayerNormalization, MultiHeadAttention,
                                     RandomContrast, Rescaling, Resizing, Reshape)
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.losses import BinaryCrossentropy,CategoricalCrossentropy, SparseCategoricalCrossentropy
from tensorflow.keras.metrics import Accuracy,TopKCategoricalAccuracy, CategoricalAccuracy, SparseCategoricalAccuracy
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import layers, models
from tensorflow.keras.regularizers  import L2, L1
import warnings
warnings.filterwarnings('ignore')
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.models import Model

<h2 style="color: BLack;">Data Collection</h2>


In [14]:
train_directory = "/kaggle/input/deepfake-and-real-images/Dataset/Train"
val_directory = "/kaggle/input/deepfake-and-real-images/Dataset/Validation"

In [15]:
CONFIGURATION = {
    "BATCH_SIZE": 32,
    "IM_SIZE":224,
    "LEARNING_RATE": 1e-3,
    "N_EPOCHS": 25,
    "DROPOUT_RATE": 0.05,
    "REGULARIZATION_RATE": 0.001,
    "N_FILTERS": 6,
    "KERNEL_SIZE": 3,
    "N_STRIDES": 1,
    "POOL_SIZE": 2,
    "N_DENSE_1": 1024,
    "N_DENSE_2": 128,
    "NUM_CLASSES": 2,
    "PATCH_SIZE": 32,
    "PROJ_DIM": 768,
    "CLASS_NAMES": ["Fake","Real"]
}

In [16]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_directory,
    labels='inferred',
    label_mode='categorical',
    class_names=CONFIGURATION["CLASS_NAMES"],
    color_mode='rgb',
    batch_size=CONFIGURATION["BATCH_SIZE"],
    image_size=(CONFIGURATION["IM_SIZE"], CONFIGURATION["IM_SIZE"]),
    shuffle=True
)

Found 140002 files belonging to 2 classes.


In [17]:
val_dataset = tf.keras.utils.image_dataset_from_directory(
    val_directory,
    labels='inferred',
    label_mode='categorical',
    class_names=CONFIGURATION["CLASS_NAMES"],
    color_mode='rgb',
    batch_size=1,#CONFIGURATION["BATCH_SIZE"],
    image_size=(CONFIGURATION["IM_SIZE"], CONFIGURATION["IM_SIZE"]),
    shuffle=True,
    seed=99,
)

Found 39428 files belonging to 2 classes.


<h2 style="color: black;">Pre-Fetch The Data For The Training Process</h2>


In [18]:
training_dataset = (
    train_dataset
    .prefetch(tf.data.AUTOTUNE)
)

In [19]:
validation_dataset = (
    val_dataset
    .prefetch(tf.data.AUTOTUNE)
)

<h2 style="color: black;">Model Building</h2>


In [20]:
input_shape = (224, 224, 3)
# Load the EfficientNetB0 model, excluding the top layers (include_top=False)
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=input_shape)
# Freeze the base model
base_model.trainable = False

# Create a Sequential model and add layers
model = models.Sequential()

# Add the EfficientNet base model
model.add(base_model)

# Add a global average pooling layer
model.add(layers.GlobalAveragePooling2D())

# Add a dropout layer for regularization
model.add(layers.Dropout(0.5))

# Add a dense layer with softmax activation for classification
model.add(layers.Dense(CONFIGURATION["NUM_CLASSES"], activation = "sigmoid"))

In [21]:
loss_function = BinaryCrossentropy()

In [22]:
model.compile(
  optimizer = Adam(learning_rate = CONFIGURATION["LEARNING_RATE"]),
  loss = loss_function,metrics=['accuracy']
)

<h2 style="color: black;">Model Training</h2>


In [ ]:
history =model.fit(
  training_dataset,
  validation_data = validation_dataset,
  batch_size= CONFIGURATION["BATCH_SIZE"],

  epochs = CONFIGURATION["N_EPOCHS"],
  verbose = 1
)

Epoch 1/25
4376/4376 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.7370 - loss: 0.5254

<h2 style="color: black;">Visualization Of The Loss</h2>


In [ ]:
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.legend(['train_loss', 'val_loss'])
plt.show()

In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('accuracy')
plt.xlabel('epoch')
plt.legend(['train_accuracy', 'val_accuracy'])
plt.show()

<h2 style="color: black;">Test The Model</h2>


In [ ]:
TESTDATA='/kaggle/input/deepfake-and-real-images/Dataset/Test'

In [ ]:
import tensorflow as tf

# Load the test dataset
test_dataset = tf.keras.utils.image_dataset_from_directory(
    TESTDATA,  # Replace with your test data directory
    image_size=(224, 224),  # Image size should match the input shape of your model
    batch_size=32,  # Batch size
    shuffle=False  # Ensure the data is not shuffled
)

In [ ]:
class_names = test_dataset.class_names
print(class_names)

In [ ]:
# Predict using the model
y_pred = model.predict(test_dataset)

# Convert predictions to class labels
y_pred_classes = tf.argmax(y_pred, axis=1)

In [ ]:
y_true = []
for images, labels in test_dataset:
    y_true.extend(labels.numpy())

y_true = tf.convert_to_tensor(y_true)

In [ ]:
from sklearn.metrics import confusion_matrix

# Compute the confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred_classes)

# Display the confusion matrix
print("Confusion Matrix:")
print(conf_matrix)

<h2 style="color: black;">Confusion Matrix</h2>


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting the confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.title('Confusion Matrix')
plt.show()

<h2 style="color: black;">Save The Model</h2>


In [ ]:
model.save('deepfake.h5')